# 📊 Chương 11: Các Khái Niệm Tương Tác (Interaction Concepts)
> **Giáo trình**: *Interactive Data Visualization* — Ward, Grinstein, Keim  
> **Tập dữ liệu**: HR Employee Attrition (IBM) — 1,470 nhân viên, 35 thuộc tính


## 0. Cài đặt & Import thư viện


In [1]:
# Cài đặt thư viện nếu chưa có: pip install plotly pandas seaborn scikit-learn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

pio.renderers.default = "notebook_connected"
print("✅ Đã import đầy đủ các thư viện!")


✅ Đã import đầy đủ các thư viện!


## 1. Đọc và khám phá dữ liệu


In [2]:
# Đọc file CSV và xem 3 dòng đầu
df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')
print(f"Tập dữ liệu: {df.shape[0]} nhân viên, {df.shape[1]} thuộc tính")
print(f"- Tỷ lệ nghỉ việc: {(df['Attrition'] == 'Yes').mean():.1%} ({(df['Attrition'] == 'Yes').sum()}/{len(df)})")
print(f"- Tuổi trung bình: {df['Age'].mean():.1f} tuổi | Thu nhập trung bình: ${df['MonthlyIncome'].mean():,.0f}/tháng")
df[['Age', 'MonthlyIncome', 'TotalWorkingYears', 'Department', 'JobRole', 'Attrition']].head(3)


Tập dữ liệu: 1470 nhân viên, 35 thuộc tính
- Tỷ lệ nghỉ việc: 16.1% (237/1470)
- Tuổi trung bình: 36.9 tuổi | Thu nhập trung bình: $6,503/tháng


,Age,MonthlyIncome,TotalWorkingYears,Department,JobRole,Attrition
0,41,5993,8,Sales,Sales Executive,Yes
1,49,5130,10,Research & Development,Research Scientist,No
2,37,2090,7,Research & Development,Laboratory Technician,Yes


---
## 2. 🧭 Điều hướng (Navigation) - Mục 11.1.1

> **Khái niệm**: Thay đổi góc nhìn, vị trí máy quay và độ chi tiết (LOD).
> - **Pan & Zoom**: Giữ chuột kéo để dịch chuyển, cuộn chuột để phóng to/thu nhỏ.
> - **Grand Tour (Asimov, 1985)**: Xoay người xem qua các góc chiếu của dữ liệu đa chiều.


In [3]:
# DEMO 1: Pan & Zoom tương tác trực tiếp
# Cuộn chuột để Zoom, nhấn giữ chuột trái để Pan

fig = px.scatter(
    df, x='Age', y='MonthlyIncome',
    color='Attrition',
    color_discrete_map={'No': '#22c55e', 'Yes': '#ef4444'},
    title='🧭 Điều hướng: Cuộn chuột để Zoom, giữ chuột trái kéo để Pan',
    labels={'Age': 'Tuổi', 'MonthlyIncome': 'Lương/tháng ($)', 'Attrition': 'Nghỉ việc'},
    opacity=0.7
)
fig.update_layout(template='plotly_dark', height=480, dragmode='pan')
fig.show(config={'scrollZoom': True})


### Demo 2: Grand Tour 3D tối giản (Mục 11.1.1 & 11.2.5)
> Đưa 3 biến thực tế vào không gian 3D. **Giữ chuột trái kéo xoay 360°** chính là thực hiện Grand Tour để quan sát mọi góc chiếu của dữ liệu.


In [4]:
# DEMO 2: Grand Tour 3D - Giữ chuột trái kéo xoay 360 độ để quét qua các góc nhìn
# 3 trục: Tuổi (X), Lương (Y), Kinh nghiệm (Z). Điểm có kích thước bằng nhau hoàn toàn (size=4).

fig_3d = px.scatter_3d(
    df,
    x='Age', y='MonthlyIncome', z='TotalWorkingYears',
    color='Attrition',
    color_discrete_map={'No': '#22c55e', 'Yes': '#ef4444'},
    labels={'Age': 'Tuổi', 'MonthlyIncome': 'Lương ($)', 'TotalWorkingYears': 'Kinh nghiệm (năm)', 'Attrition': 'Nghỉ việc'},
    title='🌐 Grand Tour 3D: Giữ chuột trái kéo xoay 360° để quan sát mọi góc chiếu',
    opacity=0.7
)
fig_3d.update_traces(marker=dict(size=4))
fig_3d.update_layout(template='plotly_dark', height=550)
fig_3d.show(config={'scrollZoom': True})
print("💡 Nhận xét: Xoay khối 3D cho thấy các điểm đỏ (nghỉ việc) dồn hẳn về góc: Tuổi trẻ, Lương thấp, Ít kinh nghiệm.")


💡 Nhận xét: Xoay khối 3D cho thấy các điểm đỏ (nghỉ việc) dồn hẳn về góc: Tuổi trẻ, Lương thấp, Ít kinh nghiệm.


---
## 3. Lựa chọn & Brushing (Selection & Brushing) — Mục 11.1.2

> **Các khía cạnh của toán tử Lựa chọn**:
> 1. **Khoanh chọn trực tiếp (Interactive Brushing)**: Kéo chuột quét một vùng dữ liệu trên biểu đồ.
> 2. **Làm nổi bật (Highlighting)**: Điểm được chọn giữ nguyên màu, các điểm còn lại mờ đi để bảo toàn bối cảnh (*Focus + Context*).
> 3. **Khung nhìn liên kết (Linked View)**: Biểu đồ phụ bên cạnh tự động cập nhật theo vùng chọn hiện tại.



In [5]:
# DEMO: Linked Brushing - Quét chuột chọn vùng -> biểu đồ cột + tỉ lệ nghỉ việc tự cập nhật
import json, html as html_mod
from IPython.display import HTML, display

data = json.dumps(df[['Age', 'MonthlyIncome', 'Department', 'Attrition']].to_dict(orient='records'))
ui = f"""
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<div id="p" style="height:380px"></div>
<div id="info" style="margin:6px 12px;padding:8px 14px;background:#1f2937;border-radius:6px;font:13px sans-serif;color:#9ca3af;border-left:3px solid #ef4444">
  Kéo chuột quét vùng trên biểu đồ trái để xem thống kê
</div>
<script>
var d = {data}, depts = ['Research & Development', 'Sales', 'Human Resources'];
var yes = d.filter(r=>r.Attrition==='Yes'), no = d.filter(r=>r.Attrition==='No');
var count = pts => depts.map(k => (pts || d).filter(p => (p.customdata || p.Department) === k).length);

Plotly.newPlot('p', [
  {{x: no.map(r=>r.Age), y: no.map(r=>r.MonthlyIncome), customdata: no.map(r=>r.Department), mode:'markers', marker:{{color:'#475569', size:4, opacity:0.4}}, name:'Ở lại', type:'scatter'}},
  {{x: yes.map(r=>r.Age), y: yes.map(r=>r.MonthlyIncome), customdata: yes.map(r=>r.Department), mode:'markers', marker:{{color:'#ef4444', size:5, opacity:0.8}}, name:'Nghỉ việc', type:'scatter'}},
  {{x: depts, y: count(), type:'bar', marker:{{color:'#ef4444'}}, name:'Phòng ban', xaxis:'x2', yaxis:'y2'}}
], {{
  template:'plotly_dark', dragmode:'select', height:380,
  margin:{{l:50, r:30, t:35, b:55}},
  grid:{{rows:1, columns:2, pattern:'independent'}},
  xaxis:{{title:'Tuổi', domain:[0,0.45]}}, yaxis:{{title:'Lương ($)'}},
  xaxis2:{{title:'Phòng ban', domain:[0.55,1], tickangle:-15}}, yaxis2:{{title:'Số lượng'}},
  annotations:[
    {{text:'Tuổi vs Lương', x:0.22, y:1.07, xref:'paper', yref:'paper', showarrow:false, font:{{size:13}}}},
    {{text:'Phòng ban', x:0.78, y:1.07, xref:'paper', yref:'paper', showarrow:false, font:{{size:13}}}}
  ]
}}).then(()=>{{
  var p = document.getElementById('p'), info = document.getElementById('info');
  p.on('plotly_selected', ev => {{
    if (!ev || !ev.points || ev.points.length===0) return;
    var pts = ev.points, total = pts.length;
    var quit = pts.filter(p=>p.curveNumber===1).length;
    var rate = (quit/total*100).toFixed(1);
    Plotly.restyle('p', {{y:[count(pts)]}}, [2]);
    info.innerHTML = 'Đã chọn <b>' + total + '</b> người | Nghỉ việc: <b style="color:#ef4444">' + quit + '</b> (' + rate + '%) | Toàn cty: 16.1%';
  }});
  p.on('plotly_deselect', ()=>{{
    Plotly.restyle('p', {{y:[count()]}}, [2]);
    info.innerHTML = 'Kéo chuột quét vùng trên biểu đồ trái để xem thống kê';
  }});
}});
</script>
"""
escaped = html_mod.escape(ui)
display(HTML(f'<iframe srcdoc="{escaped}" width="100%" height="440" style="border:none"></iframe>'))



C:\Users\Nitro\AppData\Roaming\Python\Python312\site-packages\IPython\core\display.py:448: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


---
## 4. 🔍 Lọc (Filtering) - Mục 11.1.3

> **Phân biệt Lọc vs Lựa chọn**:
> - **Lựa chọn**: Chỉ làm mờ phần không chọn, **vẫn giữ nguyên trên màn hình** để bảo toàn bối cảnh.
> - **Lọc**: **Cắt bỏ hoàn toàn** các điểm không thỏa mãn khỏi biểu đồ để giải phóng không gian hiển thị.


In [6]:
# DEMO: Lọc tương tác - Chọn thuộc tính và giá trị để lọc trực tiếp trên hình
import json, html as html_mod
from IPython.display import HTML, display

# Chuẩn bị dữ liệu
cols = ['Age', 'MonthlyIncome', 'Attrition', 'Department', 'OverTime', 'MaritalStatus', 'Gender', 'JobRole']
fdata = json.dumps(df[cols].to_dict(orient='records'))

# Các thuộc tính phân loại cho dropdown
cats = {c: df[c].unique().tolist() for c in ['Department', 'OverTime', 'MaritalStatus', 'Gender', 'JobRole']}
cats_json = json.dumps(cats)

ui = f"""
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<div style="margin:8px 0;font:13px sans-serif;color:#d1d5db;display:flex;gap:10px;align-items:center">
  <label>Thuộc tính:</label>
  <select id="attr" style="background:#1f2937;color:#f3f4f6;border:1px solid #374151;padding:4px 8px;border-radius:4px">
    <option value="OverTime">OverTime</option>
    <option value="Department">Department</option>
    <option value="MaritalStatus">MaritalStatus</option>
    <option value="Gender">Gender</option>
    <option value="JobRole">JobRole</option>
  </select>
  <label>Giá trị:</label>
  <select id="val" style="background:#1f2937;color:#f3f4f6;border:1px solid #374151;padding:4px 8px;border-radius:4px"></select>
  <span id="cnt" style="color:#9ca3af"></span>
</div>
<div id="fp" style="height:380px"></div>
<script>
var d = {fdata}, cats = {cats_json};
var attrSel = document.getElementById('attr'), valSel = document.getElementById('val'), cnt = document.getElementById('cnt');

function fillVals() {{
  valSel.innerHTML = '';
  cats[attrSel.value].forEach(v => {{ var o = document.createElement('option'); o.value = v; o.text = v; valSel.appendChild(o); }});
}}
fillVals();

function draw() {{
  var attr = attrSel.value, val = valSel.value;
  var all = d, filtered = d.filter(r => r[attr] === val);
  var color = r => r.Attrition === 'Yes' ? '#ef4444' : '#22c55e';

  Plotly.react('fp', [
    {{x: all.map(r=>r.Age), y: all.map(r=>r.MonthlyIncome), mode:'markers', marker:{{color:all.map(color), size:4, opacity:0.4}}, name:'Trước lọc', xaxis:'x', yaxis:'y'}},
    {{x: filtered.map(r=>r.Age), y: filtered.map(r=>r.MonthlyIncome), mode:'markers', marker:{{color:filtered.map(color), size:5, opacity:0.85}}, name:'Sau lọc', xaxis:'x2', yaxis:'y2'}}
  ], {{
    template:'plotly_dark', height:380, showlegend:false,
    margin:{{l:50, r:30, t:35, b:50}},
    grid:{{rows:1, columns:2, pattern:'independent'}},
    xaxis:{{title:'Tuổi', domain:[0,0.46]}}, yaxis:{{title:'Lương ($)'}},
    xaxis2:{{title:'Tuổi', domain:[0.54,1]}}, yaxis2:{{title:'Lương ($)'}},
    annotations:[
      {{text:'Trước lọc (' + all.length + ')', x:0.22, y:1.07, xref:'paper', yref:'paper', showarrow:false, font:{{size:13}}}},
      {{text:'Sau lọc ' + attr + '=' + val + ' (' + filtered.length + ')', x:0.78, y:1.07, xref:'paper', yref:'paper', showarrow:false, font:{{size:13}}}}
    ]
  }});
  var quit = filtered.filter(r=>r.Attrition==='Yes').length;
  cnt.innerHTML = 'Còn ' + filtered.length + '/' + all.length + ' | Nghỉ việc: ' + (filtered.length ? (quit/filtered.length*100).toFixed(1) : 0) + '%';
}}
draw();
attrSel.onchange = function() {{ fillVals(); draw(); }};
valSel.onchange = draw;
</script>
"""
escaped = html_mod.escape(ui)
display(HTML(f'<iframe srcdoc="{escaped}" width="100%" height="440" style="border:none"></iframe>'))



C:\Users\Nitro\AppData\Roaming\Python\Python312\site-packages\IPython\core\display.py:448: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


---
## 5. 🔄 Tái cấu hình (Reconfiguration) - Mục 11.1.4

> **Khái niệm**: Thay đổi cách bố trí không gian để tìm ra các mối quan hệ mới.
> - **Đổi phép ánh xạ**: Đổi các cặp trục X/Y khác nhau.
> - **PCA (Phân tích Thành phần Chính)**: Nén toàn bộ các biến số xuống mặt phẳng 2D.


In [7]:
# DEMO 1: Đổi phép ánh xạ trục - So sánh cặp có tương quan mạnh vs cặp không có tương quan

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Cặp 1: Kinh nghiệm vs Lương', 
                    'Cặp 2: Khoảng cách nhà vs Hài lòng']
)

# Cặp 1: TotalWorkingYears vs MonthlyIncome
fig.add_trace(go.Scatter(
    x=df['TotalWorkingYears'], y=df['MonthlyIncome'], mode='markers',
    marker=dict(color=df['Attrition'].map({'Yes': '#ef4444', 'No': '#22c55e'}), size=4, opacity=0.6),
    showlegend=False
), row=1, col=1)

# Cặp 2: DistanceFromHome vs JobSatisfaction
fig.add_trace(go.Scatter(
    x=df['DistanceFromHome'], y=df['JobSatisfaction'], mode='markers',
    marker=dict(color=df['Attrition'].map({'Yes': '#ef4444', 'No': '#22c55e'}), size=4, opacity=0.6),
    showlegend=False
), row=1, col=2)

fig.update_layout(template='plotly_dark', height=420)
fig.show(config={'scrollZoom': True})


In [8]:
# DEMO 2: PCA - Biến đổi không gian nén 15 thuộc tính số xuống 2D (PC1, PC2)

num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols = [c for c in num_cols if c not in ['EmployeeCount', 'StandardHours', 'EmployeeNumber']]

X_scaled = StandardScaler().fit_transform(df[num_cols])
pca = PCA(n_components=2)
pca_coords = pca.fit_transform(X_scaled)

df_pca = pd.DataFrame({
    'PC1': pca_coords[:, 0], 'PC2': pca_coords[:, 1],
    'Attrition': df['Attrition'], 'JobRole': df['JobRole']
})

fig = px.scatter(
    df_pca, x='PC1', y='PC2', color='Attrition',
    color_discrete_map={'No': '#22c55e', 'Yes': '#ef4444'},
    title=f'🔄 PCA: Nén {len(num_cols)} thuộc tính số xuống 2D (Giải thích {pca.explained_variance_ratio_.sum():.1%} phương sai)',
    labels={'PC1': 'Thành phần chính 1', 'PC2': 'Thành phần chính 2'},
    opacity=0.65
)
fig.update_layout(template='plotly_dark', height=450)
fig.show(config={'scrollZoom': True})


---
## 6. 🎨 Mã hóa (Encoding) - Mục 11.1.5

> **Thông điệp sách**: *"Không có một biểu đồ nào tối ưu cho mọi nhiệm vụ"*.  
> Ta có thể đổi dạng biểu đồ (Scatter, Bar, Boxplot) hoặc đổi bản đồ màu (Sequential vs Diverging).


In [9]:
# DEMO: Cùng dữ liệu Lương theo Phòng ban, 3 cách mã hóa đồ họa khác nhau

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['1. Bar Chart ', '2. Boxplot', '3. Scatter Plot (Xem từng điểm cá nhân)']
)

# 1. Bar
dept_mean = df.groupby('Department')['MonthlyIncome'].mean()
fig.add_trace(go.Bar(x=dept_mean.index, y=dept_mean.values, marker_color='#818cf8', name='Lương TB'), row=1, col=1)

# 2. Box
for dept, color in [('Sales', '#f43f5e'), ('Research & Development', '#10b981'), ('Human Resources', '#f59e0b')]:
    fig.add_trace(go.Box(y=df[df['Department']==dept]['MonthlyIncome'], name=dept, marker_color=color, showlegend=False), row=1, col=2)

# 3. Scatter
fig.add_trace(go.Scatter(x=df['Department'], y=df['MonthlyIncome'], mode='markers', marker=dict(color='#cbd5e1', size=3, opacity=0.4), showlegend=False), row=1, col=3)

fig.update_layout(template='plotly_dark', height=420, showlegend=False)
fig.show(config={'scrollZoom': True})


---
## 7. Kết nối & Khung nhìn liên kết (Connection / Linked Views) — Mục 11.1.6

> **Khái niệm**: Nhiều khung nhìn cùng hiển thị các góc cạnh khác nhau của cùng một tập dữ liệu, sử dụng chung quy tắc mã hóa màu sắc. Khi quan sát đồng thời, người dùng có thể liên kết thông tin giữa các khung nhìn để rút ra kết luận tổng hợp.



In [10]:
# DEMO: Dashboard 3 khung nhìn liên kết cùng highlight nhóm Nghỉ việc (Màu đỏ)

yes_df = df[df['Attrition'] == 'Yes']
no_df = df[df['Attrition'] == 'No']

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Tuổi x Lương', 'Tỉ lệ nghỉ việc theo Phòng', 'Độ tuổi nghỉ việc']
)

# Khung 1: Scatter
fig.add_trace(go.Scatter(x=no_df['Age'], y=no_df['MonthlyIncome'], mode='markers', marker=dict(color='#22c55e', size=3, opacity=0.3), name='Ở lại'), row=1, col=1)
fig.add_trace(go.Scatter(x=yes_df['Age'], y=yes_df['MonthlyIncome'], mode='markers', marker=dict(color='#ef4444', size=5, opacity=0.8), name='Nghỉ việc'), row=1, col=1)

# Khung 2: Bar tỉ lệ % nghỉ việc theo phòng ban
attrition_rate = df.groupby('Department')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100).round(1)
fig.add_trace(go.Bar(x=attrition_rate.index, y=attrition_rate.values, marker_color='#ef4444', text=[f'{v}%' for v in attrition_rate.values], textposition='outside', showlegend=False), row=1, col=2)

# Khung 3: Histogram phân bố tuổi
fig.add_trace(go.Histogram(x=yes_df['Age'], marker_color='#ef4444', nbinsx=15, showlegend=False), row=1, col=3)

fig.update_layout(template='plotly_dark', height=420)
fig.update_yaxes(title_text='Tỉ lệ (%)', row=1, col=2)
fig.show(config={'scrollZoom': True})



---
## 8. 🔍 Khái quát hóa & Chi tiết hóa (Abstraction / Elaboration) - Mục 11.1.7

> **Overview + Detail**: Phía trên nhìn toàn cảnh (Khái quát hóa), phía dưới phóng to xem sâu vào nhóm tiêu điểm (Chi tiết hóa).


In [11]:
# DEMO: Overview + Detail (Khái quát hóa kết hợp Chi tiết hóa)

fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.35, 0.65],
    subplot_titles=['1. OVERVIEW: Toàn bộ 1,470 nhân viên (Khung tím là vùng tiêu điểm 25-35 tuổi)',
                    '2. DETAIL: Phóng to chi tiết nhóm 25-35 tuổi (Thấy rõ từng cá nhân)']
)

# 1. Overview
fig.add_trace(go.Scatter(x=df['Age'], y=df['MonthlyIncome'], mode='markers', marker=dict(color='#64748b', size=3, opacity=0.3), showlegend=False), row=1, col=1)
fig.add_vrect(x0=25, x1=35, fillcolor='#818cf8', opacity=0.2, line_width=1.5, line_color='#818cf8', row=1, col=1)

# 2. Detail
focus = df[(df['Age'] >= 25) & (df['Age'] <= 35)]
fig.add_trace(go.Scatter(
    x=focus['Age'], y=focus['MonthlyIncome'], mode='markers',
    marker=dict(color=focus['Attrition'].map({'Yes': '#ef4444', 'No': '#22c55e'}), size=6, opacity=0.8),
    showlegend=False
), row=2, col=1)

fig.update_layout(template='plotly_dark', height=550)
fig.show(config={'scrollZoom': True})


---
## 9. ⚙️ Khuôn khổ thống nhất (Unified Framework) - Mục 11.3

> Bộ ba tương tác: $\text{Interaction} = \langle \text{Toán tử}, \text{Không gian}, \text{Tham số} \rangle$.
> Ví dụ: **Pipeline kết hợp chuỗi Toán tử**: Lọc ➔ Tái cấu hình (PCA) ➔ Mã hóa ➔ Hiển thị.


In [12]:
# DEMO: Pipeline kết hợp chuỗi 3 toán tử
# Bước 1: Lọc (Chỉ lấy phòng Sales và R&D)
step1 = df[df['Department'].isin(['Sales', 'Research & Development'])].copy()

# Bước 2: Tái cấu hình (PCA giảm chiều)
features = ['Age', 'MonthlyIncome', 'TotalWorkingYears', 'YearsAtCompany']
pca_res = PCA(n_components=2).fit_transform(StandardScaler().fit_transform(step1[features]))
step1['PC1'] = pca_res[:, 0]
step1['PC2'] = pca_res[:, 1]

# Bước 3 & 4: Mã hóa và Hiển thị
fig = px.scatter(
    step1, x='PC1', y='PC2',
    color='Department', symbol='Attrition',
    color_discrete_map={'Sales': '#f43f5e', 'Research & Development': '#10b981'},
    title='⚙️ Pipeline: Lọc (Sales & R&D) ➔ Tái cấu hình (PCA) ➔ Mã hóa (Màu & Ký hiệu)',
    opacity=0.7
)
fig.update_layout(template='plotly_dark', height=480)
fig.show(config={'scrollZoom': True})

---
## 10. Bảng tổng hợp các Toán tử Tương tác — Chương 11

| Toán tử | Mục trong sách | Định nghĩa cốt lõi | Minh họa trong notebook |
|---|---|---|---|
| **1. Điều hướng (Navigation)** | 11.1.1 | Thay đổi vị trí máy quay, hướng nhìn | Pan, Zoom, Grand Tour 3D xoay 360° |
| **2. Lựa chọn (Selection)** | 11.1.2 | Cô lập tập con dữ liệu để phân tích | Linked Brushing: quét chọn vùng, biểu đồ phụ tự cập nhật |
| **3. Lọc (Filtering)** | 11.1.3 | Loại bỏ hoàn toàn các điểm không thỏa mãn | Chọn thuộc tính và giá trị qua dropdown, so sánh trước/sau |
| **4. Tái cấu hình (Reconfiguration)** | 11.1.4 | Thay đổi cách bố trí không gian | Đổi cặp trục tọa độ, PCA giảm chiều |
| **5. Mã hóa (Encoding)** | 11.1.5 | Đổi cách biểu diễn đồ họa | Đổi dạng biểu đồ (Scatter, Bar, Boxplot) |
| **6. Kết nối (Connection)** | 11.1.6 | Liên kết giữa các khung nhìn | 3 khung nhìn cùng mã hóa màu chung nhóm Nghỉ việc |
| **7. Khái quát / Chi tiết** | 11.1.7 | Cân bằng giữa tổng thể và chi tiết | Overview toàn cảnh + Detail phóng to nhóm tiêu điểm |
| **8. Khuôn khổ thống nhất** | 11.3 | Ghép chuỗi các toán tử thành quy trình | Pipeline: Lọc → Tái cấu hình → Mã hóa → Hiển thị |

